# Southwest Airlines Data Preprocessing

## To do
- Validate the dataset (and clean if neccessary)
- Check missing values and outliers
- Create new features
- Prepare data for machine learning

In [ ]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../dataset/raw_data.csv")

print("\nData types:")
print(df.dtypes)

print("\nMissing values check:")
missing_values = df.isnull().sum()
if missing_values.sum() == 0:
    print("No missing values")
else:
    print(missing_values[missing_values > 0])



Data types:
year                     int64
month                    int64
carrier                 object
carrier_name            object
airport                 object
airport_name            object
arr_flights            float64
arr_del15              float64
carrier_ct             float64
weather_ct             float64
nas_ct                 float64
security_ct            float64
late_aircraft_ct       float64
arr_cancelled          float64
arr_diverted           float64
arr_delay              float64
carrier_delay          float64
weather_delay          float64
nas_delay              float64
security_delay         float64
late_aircraft_delay    float64
dtype: object

Missing values check:
No missing values


## Check if data is intuitive
- Do outlier check

In [4]:
print(f" Total airports: {df['airport'].nunique()}")


 Total airports: 113


In [5]:
# use IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    
    return outliers, lower_bound, upper_bound

key_columns = ['arr_flights', 'arr_del15', 'arr_delay']

for col in key_columns:
    outliers, lower_bound, upper_bound = detect_outliers_iqr(df, col)
    outlier_count = len(outliers)
    outlier_percentage = (outlier_count / len(df)) * 100
    
    print(f"\n{col}:")
    print(f"  Outliers: {outlier_count} ({outlier_percentage:.2f}%)")
    print(f"  Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    if outlier_count > 0:
        print(f"  Min outlier: {outliers[col].min():.2f}")
        print(f"  Max outlier: {outliers[col].max():.2f}")



arr_flights:
  Outliers: 1371 (12.34%)
  Bounds: [-1389.50, 2894.50]
  Min outlier: 2896.00
  Max outlier: 8727.00

arr_del15:
  Outliers: 1192 (10.73%)
  Bounds: [-285.00, 579.00]
  Min outlier: 580.00
  Max outlier: 3037.00

arr_delay:
  Outliers: 1161 (10.45%)
  Bounds: [-14738.00, 29438.00]
  Min outlier: 29546.00
  Max outlier: 194272.00


In [8]:
def preprocess_southwest_data(df):
    print(f"Original shape: {df.shape}")
    
    df = df.copy()
    
    # data type converting
    print("\n Converting data types...")
    df['year'] = df['year'].astype(int)
    df['month'] = df['month'].astype(int)
    
    # create features
    print("\n creating features...")
    
    # delay rate (percentage of delayed flights)
    df['delay_rate'] = df['arr_del15'] / df['arr_flights']
    
    # cancellation rate
    df['cancellation_rate'] = df['arr_cancelled'] / df['arr_flights']
    
    # average delay time
    df['avg_delay_time'] = df['arr_delay'] / df['arr_del15']
    df['avg_delay_time'] = df['avg_delay_time'].fillna(0)
    
    # delay cause rates
    delay_causes = ['carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct']
    for cause in delay_causes:
        rate_col = f"{cause}_rate"
        df[rate_col] = df[cause] / df['arr_flights']
    
    # seasonal categorization
    def get_season(month):
        if month in [12, 1, 2]:
            return 'Winter'
        elif month in [3, 4, 5]:
            return 'Spring'
        elif month in [6, 7, 8]:
            return 'Summer'
        else:
            return 'Fall'
    
    df['season'] = df['month'].apply(get_season)
    
    # datetime for time series analysis
    df['year_month'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
    
    # airport size categorization
    airport_flights = df.groupby('airport')['arr_flights'].mean()
    small_threshold = airport_flights.quantile(0.33)
    large_threshold = airport_flights.quantile(0.67)
    
    def categorize_airport_size(airport):
        avg_flights = airport_flights[airport]
        if avg_flights <= small_threshold:
            return 'Small'
        elif avg_flights <= large_threshold:
            return 'Medium'
        else:
            return 'Large'
    
    df['airport_size'] = df['airport'].apply(categorize_airport_size)
    
    # handle outliers
    print("\n handling outliers...")
    
    outlier_columns = ['arr_flights', 'arr_delay', 'avg_delay_time']
    for col in outlier_columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        upper_bound = Q3 + 1.5 * IQR
        
        outliers_mask = df[col] > upper_bound
        outlier_count = outliers_mask.sum()
        
        if outlier_count > 0:
            df.loc[outliers_mask, col] = upper_bound
            print(f"{col}: Capped {outlier_count} outliers at {upper_bound:.2f}")
    
    return df

# preprocess it all
df_final = preprocess_southwest_data(df)

Original shape: (11109, 21)

 Converting data types...

 creating features...

 handling outliers...
arr_flights: Capped 1371 outliers at 2894.50
arr_delay: Capped 1161 outliers at 29438.00
avg_delay_time: Capped 182 outliers at 71.11


## Validate preprocessing

In [9]:
new_features = ['delay_rate', 'cancellation_rate', 'avg_delay_time']
print(f"\nSummary statistics for new features")
print(df_final[new_features].describe())

# Season distribution
print(f"\nSeason distribution:")
print(df_final['season'].value_counts())

# Airport size distribution
print(f"\nAirport size distribution:")
print(df_final['airport_size'].value_counts())

# Sample of final dataset
print(f"\nSample of final dataset:")
print(df_final.head())



Summary statistics for new features
         delay_rate  cancellation_rate  avg_delay_time
count  11109.000000       11109.000000    11109.000000
mean       0.205420           0.021133       48.419680
std        0.092726           0.052166        9.222757
min        0.000000           0.000000        0.000000
25%        0.143564           0.002457       42.740741
50%        0.201717           0.008152       47.774775
75%        0.263754           0.019928       54.090177
max        0.628415           0.703333       71.114332

Season distribution:
season
Summer    2868
Spring    2772
Winter    2746
Fall      2723
Name: count, dtype: int64

Airport size distribution:
airport_size
Large     4410
Medium    3870
Small     2829
Name: count, dtype: int64

Sample of final dataset:
   year  month carrier            carrier_name airport  \
0  2023      8      WN  Southwest Airlines Co.     ABQ   
1  2023      8      WN  Southwest Airlines Co.     ALB   
2  2023      8      WN  Southwest Airline

# Save preprocessed csv

In [10]:
from pathlib import Path

output = Path("../dataset/preprocessed.csv")
df_final.to_csv(output, index=False)

print(f"original dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"final dataset: {df_final.shape[0]} rows, {df_final.shape[1]} columns")
print(f"features added: {df_final.shape[1] - df.shape[1]}")


original dataset: 11109 rows, 21 columns
final dataset: 11109 rows, 32 columns
features added: 11


## Some insights

In [11]:
# Overall statistics
overall_delay_rate = df_final['delay_rate'].mean()
print(f"Overall average delay rate: {overall_delay_rate:.2%}")

# Seasonal patterns
seasonal_delays = df_final.groupby('season')['delay_rate'].mean().sort_values(ascending=False)
worst_season = seasonal_delays.index[0]
best_season = seasonal_delays.index[-1]
print(f"Worst season for delays: {worst_season} ({seasonal_delays[worst_season]:.2%})")
print(f"Best season for delays: {best_season} ({seasonal_delays[best_season]:.2%})")

# Airport size patterns
size_delays = df_final.groupby('airport_size')['delay_rate'].mean().sort_values(ascending=False)
worst_size = size_delays.index[0]
print(f"Worst airport size for delays: {worst_size} ({size_delays[worst_size]:.2%})")

# Worst airport
airport_delays = df_final.groupby('airport')['delay_rate'].mean().sort_values(ascending=False)
worst_airport = airport_delays.index[0]
print(f"Worst airport: {worst_airport} ({airport_delays[worst_airport]:.2%})")


Overall average delay rate: 20.54%
Worst season for delays: Summer (25.59%)
Best season for delays: Fall (16.28%)
Worst airport size for delays: Medium (21.32%)
Worst airport: EWR (30.78%)


## ML Feature Selection Analysis

Let's analyze which columns can be dropped for machine learning purposes, since we've compressed information into new features.


In [12]:
print(f"Current dataset shape: {df_final.shape}")
print(f"Columns: {list(df_final.columns)}")

# categorize columns

# identifiers (can be dropped for ML)
identifiers = ['carrier', 'carrier_name', 'airport', 'airport_name']
print(f"Identifiers: {identifiers}")

# raw counts (redundant with rates)
raw_counts = ['arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted']
print(f"Raw counts: {raw_counts}")

# raw delays (redundant with avg_delay_time)
raw_delays = ['arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']
print(f"Raw delays : {raw_delays}")

# keep for machine learning
keep_for_ml = ['arr_flights', 'delay_rate', 'cancellation_rate', 'avg_delay_time', 
               'carrier_ct_rate', 'weather_ct_rate', 'nas_ct_rate', 'security_ct_rate', 'late_aircraft_ct_rate',
               'season', 'airport_size']

print(f"Keep for ML: {keep_for_ml}")

# optional (depends on ML task)
optional = ['year_month']  # Can be useful for time series
print(f"Optional (depends on task): {optional}")

# calculate redundancy
total_columns = len(df_final.columns)
redundant_columns = len(identifiers) + len(raw_counts) + len(raw_delays)
keep_columns = len(keep_for_ml)

print(f"Total columns: {total_columns}")
print(f"Redundant columns: {redundant_columns}")
print(f"Essential columns: {keep_columns}")
print(f"Reduction: {redundant_columns}/{total_columns} = {redundant_columns/total_columns:.1%} reduction")


Current dataset shape: (11109, 32)
Columns: ['year', 'month', 'carrier', 'carrier_name', 'airport', 'airport_name', 'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted', 'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay', 'delay_rate', 'cancellation_rate', 'avg_delay_time', 'carrier_ct_rate', 'weather_ct_rate', 'nas_ct_rate', 'security_ct_rate', 'late_aircraft_ct_rate', 'season', 'year_month', 'airport_size']
Identifiers: ['carrier', 'carrier_name', 'airport', 'airport_name']
Raw counts: ['arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted']
Raw delays : ['arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']
Keep for ML: ['arr_flights', 'delay_rate', 'cancellation_rate', 'avg_delay_time', 'carrier_ct_rate', 'weather_ct_rate', 'nas_ct_rate', 'securit

## Feature selection

In [13]:
def create_ml_dataset_with_airport(df):
    ml_columns = ['airport', 'arr_flights', 'delay_rate', 'cancellation_rate', 'avg_delay_time', 
                  'carrier_ct_rate', 'weather_ct_rate', 'nas_ct_rate', 'security_ct_rate', 'late_aircraft_ct_rate',
                  'season', 'airport_size']
    
    df_ml = df[ml_columns].copy()
    
    print(f"ML dataset with airport created:")
    print(f"  Original shape: {df.shape}")
    print(f"  ML shape: {df_ml.shape}")
    print(f"  Columns reduced: {df.shape[1] - df_ml.shape[1]}")
    print(f"  Size reduction: {(df.shape[1] - df_ml.shape[1])/df.shape[1]:.1%}")
    
    return df_ml

# create ML dataset including airport
df_ml_with_airport = create_ml_dataset_with_airport(df_final)

print("Features kept for ML:")
for i, col in enumerate(df_ml_with_airport.columns, 1):
    print(f"  {i:2d}. {col}")

print("Numerical features:")
numerical_features = df_ml_with_airport.select_dtypes(include=[np.number]).columns.tolist()
for feat in numerical_features:
    print(f"  - {feat}")

print("\nCategorical features:")
categorical_features = df_ml_with_airport.select_dtypes(include=['object']).columns.tolist()
for feat in categorical_features:
    unique_count = df_ml_with_airport[feat].nunique()
    print(f"  - {feat} (unique values: {unique_count})")
    if feat == 'airport':
        print(f"    Sample airports: {df_ml_with_airport[feat].unique()[:10].tolist()}")

ML dataset with airport created:
  Original shape: (11109, 32)
  ML shape: (11109, 12)
  Columns reduced: 20
  Size reduction: 62.5%
Features kept for ML:
   1. airport
   2. arr_flights
   3. delay_rate
   4. cancellation_rate
   5. avg_delay_time
   6. carrier_ct_rate
   7. weather_ct_rate
   8. nas_ct_rate
   9. security_ct_rate
  10. late_aircraft_ct_rate
  11. season
  12. airport_size
Numerical features:
  - arr_flights
  - delay_rate
  - cancellation_rate
  - avg_delay_time
  - carrier_ct_rate
  - weather_ct_rate
  - nas_ct_rate
  - security_ct_rate
  - late_aircraft_ct_rate

Categorical features:
  - airport (unique values: 113)
    Sample airports: ['ABQ', 'ALB', 'AMA', 'ATL', 'AUS', 'BDL', 'BHM', 'BLI', 'BNA', 'BOI']
  - season (unique values: 4)
  - airport_size (unique values: 3)


## Save updated dataset, ready for modeling

In [14]:
ml_output_path = Path("../dataset/ml_ready.csv")
df_ml_with_airport.to_csv(ml_output_path, index=False)

## Airport specific patterns

In [15]:

# Top 10 airports by delay rate
top_delay_airports = df_ml_with_airport.groupby('airport')['delay_rate'].mean().sort_values(ascending=False).head(10)
print("Top 10 airports by average delay rate:")
for i, (airport, rate) in enumerate(top_delay_airports.items(), 1):
    print(f"  {i:2d}. {airport}: {rate:.2%}")

print(f"\nAirport size vs delay rate:")
size_delay = df_ml_with_airport.groupby('airport_size')['delay_rate'].mean().sort_values(ascending=False)
for size, rate in size_delay.items():
    print(f"  {size}: {rate:.2%}")

print(f"\nSeasonal patterns by airport size:")
seasonal_patterns = df_ml_with_airport.groupby(['airport_size', 'season'])['delay_rate'].mean().unstack()
print(seasonal_patterns.round(3))

Top 10 airports by average delay rate:
   1. EWR: 30.78%
   2. FAT: 29.14%
   3. ORD: 27.86%
   4. SFO: 27.50%
   5. SYR: 27.47%
   6. COS: 26.10%
   7. MIA: 25.45%
   8. BZN: 25.40%
   9. JAN: 25.16%
  10. MYR: 24.20%

Airport size vs delay rate:
  Medium: 21.32%
  Small: 21.10%
  Large: 19.50%

Seasonal patterns by airport size:
season         Fall  Spring  Summer  Winter
airport_size                               
Large         0.157   0.190   0.235   0.197
Medium        0.169   0.205   0.270   0.206
Small         0.165   0.205   0.270   0.200
